# B2 · Tabloya yeni sütun eklemek

**Bugünün tek fikri:** var olan sütunlardan **yeni bir sütun türetmek**: `tablo["yeni"] = <ifade>`. Cuma günkü B3'te (groupby) gruplayacağın sütunları (mevsim, saat, rüzgâr sınıfı) bugün kendin üreteceksin.

**Kural (B1'deki gibi):** her kod hücresinden önce altındaki kutuya "bu ne yapıyor" diye tek cümle yaz. Sonra çalıştırıp kontrol et.

## 1. Oyuncak tablo

Gene 5 satır. `olcum` ve `gfs` knot cinsinden, `saat_utc` 0–23 arası saat, `ay` 1–12 arası.

In [1]:
import pandas as pd

oyuncak = pd.DataFrame({
    "olcum":    [4, 12, 18, 22, 9],
    "gfs":      [5, 8, 10, 21, 9],
    "saat_utc": [2, 9, 12, 22, 23],
    "ay":       [1, 4, 7, 7, 11],
})
oyuncak

,olcum,gfs,saat_utc,ay
0,4,5,2,1
1,12,8,9,4
2,18,10,12,7
3,22,21,22,7
4,9,9,23,11


✍️ *Oyuncak adında bir tablo oluşturdu 4 tane özelliği olan*



### 1a. İki sütundan yeni sütun

`oyuncak["olcum"] - oyuncak["gfs"]` her satırda **kendi** değerleri birbirinden çıkarır. Döngü yazmana gerek yok. Sonucu `oyuncak["fark"] = ...` ile tabloya yeni bir sütun olarak yazıyorsun.

In [2]:
oyuncak["fark"] = oyuncak["gfs"] - oyuncak["olcum"]
oyuncak

,olcum,gfs,saat_utc,ay,fark
0,4,5,2,1,1
1,12,8,9,4,-4
2,18,10,12,7,-8
3,22,21,22,7,-1
4,9,9,23,11,0


fark adında yeni bir sütun açtı tahmini ile gerçek arasındaki farkı veren

### 1b. Maske de bir sütun olabilir

B1'deki maske (True/False Series) aynen tabloya yazılabiliyor. İleride "kuvvetli saatler" diye sürekli yeniden hesaplamak yerine sütundan okursun.

In [3]:
oyuncak["kuvvetli"] = oyuncak["olcum"] >= 15
oyuncak

,olcum,gfs,saat_utc,ay,fark,kuvvetli
0,4,5,2,1,1,False
1,12,8,9,4,-4,False
2,18,10,12,7,-8,True
3,22,21,22,7,-1,True
4,9,9,23,11,0,False


Maskeyi sütun yapıp tabloya ekledi

### 1c. Sütun üstünde hesap: `+`, `%`, `.abs()`

UTC'yi İzmir saatine çevirmek: +3 ekle, 24'e göre kalanını al (`%`). 22+3=25 olunca 1'e döner.

In [4]:
oyuncak["yerel_saat"] = (oyuncak["saat_utc"] + 3) % 24
oyuncak["mutlak_fark"] = oyuncak["fark"].abs()
oyuncak

,olcum,gfs,saat_utc,ay,fark,kuvvetli,yerel_saat,mutlak_fark
0,4,5,2,1,1,False,5,1
1,12,8,9,4,-4,False,12,4
2,18,10,12,7,-8,True,15,8
3,22,21,22,7,-1,True,1,1
4,9,9,23,11,0,False,2,0


yerel saat ve mutlak farkı ekledi

### 1d. Sözlükle çevirmek: `.map()`

Her değeri bir sözlükten karşılığıyla değiştirir. `ay` sütunundaki 7, sözlükteki `7: "yaz"` ile `"yaz"` olur.

In [5]:
mevsim_sozluk = {12: "kis", 1: "kis", 2: "kis",
                 3: "ilkbahar", 4: "ilkbahar", 5: "ilkbahar",
                 6: "yaz", 7: "yaz", 8: "yaz",
                 9: "sonbahar", 10: "sonbahar", 11: "sonbahar"}
oyuncak["mevsim"] = oyuncak["ay"].map(mevsim_sozluk)
oyuncak

,olcum,gfs,saat_utc,ay,fark,kuvvetli,yerel_saat,mutlak_fark,mevsim
0,4,5,2,1,1,False,5,1,kis
1,12,8,9,4,-4,False,12,4,ilkbahar
2,18,10,12,7,-8,True,15,8,yaz
3,22,21,22,7,-1,True,1,1,yaz
4,9,9,23,11,0,False,2,0,sonbahar


Mevsim sözlüpü oluşturdu daha sonra tablodaki aya göre uygun mevsime atayıp tabloya ekledi.(bunu tek tek yazmak yerine 1den3e kadar kış vb yazabiliryor muyu<?>)

In [7]:
#üsteki soruma cevap alttaki cut sanırım

### 1e. Sayıyı aralığa çevirmek: `pd.cut`

`bins` sınırları verir. `right=False` olunca aralık `[10, 15)` olur, yani 10 dahil, 15 hariç. `labels` her aralığın adı.

In [8]:
oyuncak["sinif"] = pd.cut(oyuncak["olcum"],
                         bins=[0, 10, 15, 20, 100],
                         labels=["hafif", "orta", "iyi", "kuvvetli"],
                         right=False)
oyuncak

,olcum,gfs,saat_utc,ay,fark,kuvvetli,yerel_saat,mutlak_fark,mevsim,sinif
0,4,5,2,1,1,False,5,1,kis,hafif
1,12,8,9,4,-4,False,12,4,ilkbahar,orta
2,18,10,12,7,-8,True,15,8,yaz,iyi
3,22,21,22,7,-1,True,1,1,yaz,kuvvetli
4,9,9,23,11,0,False,2,0,sonbahar,hafif


Çok hoşuma gitti aslıdna yukarıda sözlükle yaptığımız şeyi burada basitçe aralık tanımlayarak ve bu aralıklara isimler vererek halletti

### 1f. Yeni sütuna bakmak: `.value_counts()` ve `.sort_values()`

`value_counts`: her değerden kaç tane var. `sort_values`: tabloyu bir sütuna göre sırala (`ascending=False` büyükten küçüğe).

In [9]:
print(oyuncak["mevsim"].value_counts())
oyuncak.sort_values("mutlak_fark", ascending=False)

mevsim
yaz         2
kis         1
ilkbahar    1
sonbahar    1
Name: count, dtype: int64


,olcum,gfs,saat_utc,ay,fark,kuvvetli,yerel_saat,mutlak_fark,mevsim,sinif
2,18,10,12,7,-8,True,15,8,yaz,iyi
1,12,8,9,4,-4,False,12,4,ilkbahar,orta
0,4,5,2,1,1,False,5,1,kis,hafif
3,22,21,22,7,-1,True,1,1,yaz,kuvvetli
4,9,9,23,11,0,False,2,0,sonbahar,hafif


İlki o sütundaki değerlerin kaç kere geçtiğini verir. ikincisi ise belli bir sütuna göre sıralama yapar

## 2. Gerçek veri

Şimdi aynı şeyleri sen yazıyorsun. Her hücrede iskelet ve yorum satırı var. Takılırsan önce yukarıdaki oyuncak hücreye bak, sonra Pezo'ya ipucu sor.

In [10]:
from ruzgar.data import load_aligned
veri = load_aligned()
veri.head()

,time,speed_kn_gfs,dir_deg_gfs,gust_kn_gfs,station,speed_kn_obs,dir_deg_obs,gust_kn_obs,hata
0,2024-03-01 01:00:00+00:00,1.8,139,2.1,LTBJ,2.0,NaN,NaN,-0.2
1,2024-03-01 02:00:00+00:00,2.0,287,1.6,LTBJ,5.0,150.0,NaN,-3.0
2,2024-03-01 03:00:00+00:00,1.6,346,1.7,LTBJ,2.0,160.0,NaN,-0.4
3,2024-03-01 04:00:00+00:00,0.4,297,0.4,LTBJ,6.0,350.0,NaN,-5.6
4,2024-03-01 05:00:00+00:00,0.9,207,0.6,LTBJ,4.0,350.0,NaN,-3.1


### Alıştırma 1 · `hata` sütununu kendin üret

Tabloda zaten `hata` diye bir sütun var. İlk satıra bak: gfs 1,8, ölçüm 2,0, hata −0,2. **Önce** hangi çıkarma olduğunu tahmin et, sonra `benim_hata` sütununu yaz ve ikisinin aynı olduğunu kontrol et (en büyük fark ~0 çıkmalı).

In [11]:
veri["benim_hata"] = veri["speed_kn_gfs"]- veri["speed_kn_obs"]
(veri["benim_hata"] - veri["hata"]).abs().max()


np.float64(0.0)

### Alıştırma 2 · Ham GFS'in ortalama mutlak hatası

`mutlak_hata` sütunu ekle (`.abs()`), sonra ortalamasını al. Bu sayı, rüzgâr projesindeki "ham GFS" MAE'sinin bütün veri üstündeki hali.

In [12]:
veri["mutlak_hata"] = veri["benim_hata"].abs()
veri["mutlak_hata"].mean()


np.float64(4.4677207357145905)

### Alıştırma 3 · Yerel saat ve kontrol

`yerel_saat` sütunu ekle (`veri["time"].dt.hour` ile başla). Sonra `value_counts()` ile her saatten kaç tane olduğuna bak. Hepsi yaklaşık aynı mı? Neden öyle olmalı?

In [17]:
veri["yerel_saat"] = ((veri["time"].dt.hour +3) % 24)
veri["yerel_saat"].value_counts().sort_index()


yerel_saat
0     977
1     977
2     977
3     976
4     977
5     977
6     976
7     976
8     976
9     977
10    976
11    977
12    977
13    975
14    976
15    976
16    977
17    977
18    976
19    976
20    976
21    976
22    976
23    976
Name: count, dtype: int64

### Alıştırma 4 · Mevsim

`mevsim` sütununu ekle (`.dt.month` + yukarıdaki `mevsim_sozluk`). Her mevsimden kaç saat var? Veri Mart 2024'te başlıyor; sayılar eşit değilse nedeni bu olabilir mi?

In [18]:
veri["mevsim"] = (veri["time"].dt.month).map(mevsim_sozluk)
veri["mevsim"].value_counts()


mevsim
ilkbahar    6623
yaz         6622
sonbahar    5876
kis         4312
Name: count, dtype: int64

### Alıştırma 5 · Rüzgâr sınıfı

`sinif` sütununu **ölçülen** hızdan `pd.cut` ile üret (aynı bins ve labels). Kaç saat "kuvvetli"? B1'deki 789 ile karşılaştır.

In [20]:
veri["sinif"] = pd.cut(
    veri["speed_kn_obs"],
    bins=[0, 10, 15, 20, 100],
    labels=["hafif", "orta", "iyi", "kuvvetli"],
    right=False)
veri["sinif"].value_counts()


sinif
hafif       12469
orta         5799
iyi          4376
kuvvetli      789
Name: count, dtype: int64

### Alıştırma 6 · GFS'in en kötü 5 saati

`mutlak_hata`'ya göre büyükten küçüğe sırala, ilk 5 satırda yalnız `time`, `speed_kn_obs`, `speed_kn_gfs`, `hata` sütunlarını göster. Bu saatlerde GFS rüzgârı fazla mı tahmin etmiş, eksik mi?

In [27]:
veri.sort_values("mutlak_hata", ascending=False).head(5)[["time", "speed_kn_obs", "speed_kn_gfs", "hata"]]


,time,speed_kn_obs,speed_kn_gfs,hata
4060,2024-08-17 05:00:00+00:00,26.0,5.1,-20.9
13862,2025-09-30 01:00:00+00:00,23.0,2.9,-20.1
22985,2026-10-15 07:00:00+00:00,29.0,9.1,-19.9
7217,2024-12-27 02:00:00+00:00,24.0,4.6,-19.4
18687,2026-04-19 02:00:00+00:00,24.0,5.0,-19.0


## 3. Kapanış (2 cümle)

✍️ *Bugün öğrendiğim tek şey:*
bugün tablo birleştirme, sütun açma, hesap yapma kategorize etme vb. öğrendim güzel oldu.

✍️ *Cuma B3'te bu sütunlarla hangi soruyu sormak isterdim? (ör. "hangi mevsimde GFS en çok yanılıyor?")*
Cuma günü bu verilerle yararlı sonuçlar üretmek istiyorum dediğin gib modelin öğrenmesi için hangi mevsimde hangi saatlerde en çok şaşıyor en tutarlı tahminleri hafifte mi yapıyor vb